# Controlled Synthetic Study of Spatiotemporal Traffic Forecasting Models

## 1. Purpose of This Notebook

This notebook contains a controlled, simulation-based evaluation of the forecasting models studied in our PeMS work. The goal is **not** to replace the real-data results, but to provide a controlled testbed where we can vary the data-generating process and observe how each model responds. By doing so, we can move beyond statements of the form *"model A beat model B on this dataset"* toward statements of the form *"model A beats model B when condition X holds, and we can verify that condition X is present in the real-world data."*

In our PeMS experiments, we observed three patterns that are difficult to interpret on real data alone:

1. Random Forest achieved the lowest MAE under clean, full-information conditions, despite ignoring spatial structure entirely.
2. Under simulated sensor outages, GraphWaveNet–LSTM outperformed the proposed GraphWaveNet–GRU–LSTM, while GraphWaveNet–GRU performed substantially worse than either.
3. The proposed model improved over the base GraphWaveNet at every horizon, but the magnitude of the improvement was modest at long horizons (≈3.4% at 72 h).

Each of these observations admits multiple explanations, and real data does not let us isolate them. For example, RF may be winning because (a) the spatial coupling between stations is genuinely weak in District 3, (b) our 24-hour input window prevents graph models from accessing weekly history that RF can engineer as features, (c) the training-set size is too small for graph models to learn well, or (d) some combination of these. The simulation study is designed to disentangle these explanations.

## 2. Research Questions

We organize the simulation study around five research questions, each addressed by a dedicated experiment. The first three are required for the publication-grade version of this study; the remaining two strengthen the conclusions but can be reduced in scope if computational budget is constrained.

**RQ1 — Spatial coupling.** How does the relative performance of feature-based models (Random Forest), non-graph deep models, and graph-based models depend on the strength of spatial coupling between stations? We hypothesize that there exists a coupling threshold below which graph methods cannot outperform feature-based baselines, and that District 3 falls near or below this threshold.

**RQ2 — Outage typology.** Real sensor failures are rarely independent across stations or windows. They tend to be spatially clustered (an entire corridor goes dark when a roadside cabinet fails), temporally extended (a station is offline for days, not hours), or intermittent within a window. Are the robustness conclusions drawn from random per-window masking representative of more realistic failure modes?

**RQ3 — Training-seed variance.** The robustness comparisons in our PeMS study used a single trained checkpoint per architecture, with variance reported only across mask seeds. Are the observed differences between graph variants (in particular the gap between GraphWaveNet–GRU and GraphWaveNet–LSTM under outage) statistically meaningful, or are they within the range of training-seed noise?

**RQ4 — Input-window length.** Does the gap between Random Forest and graph models close when graph models are given access to longer input histories that include weekly periodic structure?

**RQ5 — Horizon-refinement interaction.** Does the benefit of stacked GRU–LSTM refinement grow with forecast horizon, as the architecture motivation in the paper suggests, or is it horizon-independent?

## 3. Approach Overview

We construct a synthetic traffic generator with explicit, controllable knobs for the quantities the research questions ask about. The generator produces hourly flow series on a graph of sensors arranged along corridors, in the same NumPy format as our real-data pipeline (`X`, `Y`, `A`, `train_starts`, `val_starts`, `test_starts`, etc.), so that **the existing model code runs on simulated data without modification**.

For each experiment, we vary one knob of the generator while holding the others fixed at calibrated defaults, train each model on the resulting dataset, and evaluate on a held-out test split. All splits use the same target-window-contained logic as the PeMS pipeline to prevent leakage.

The models evaluated are:

- Elastic Net (regularized linear regression baseline)
- Random Forest (tree ensemble baseline, per-station)
- LSTM (non-graph deep baseline)
- CNN-GRU-LSTM (non-graph hybrid baseline)
- GraphWaveNet (graph baseline)
- GraphWaveNet-GRU
- GraphWaveNet-LSTM
- GraphWaveNet-GRU-LSTM (proposed)

Hyperparameters for each model are fixed across experiments at the values used in the PeMS study. This is a deliberate choice: re-tuning per experiment would conflate architectural advantages with hyperparameter-search advantages.

## 4. Synthetic Data-Generating Process

We define a controllable synthetic flow generator $\mathcal{G}(\theta)$ where $\theta$ is the vector of generation parameters. The graph $G = (V, E)$ has $N$ nodes arranged along $C$ corridors, mimicking the freeway-corridor topology of PeMS. Each node $n$ is connected to its $K$ nearest upstream and $K$ nearest downstream neighbors on the same corridor, with edge weights given by a Gaussian kernel of post-mile distance, exactly as in the PeMS adjacency construction.

For each node $n \in \{1, \dots, N\}$ and each hour $t \in \{1, \dots, T\}$, the observed flow is generated in three additive components: a **deterministic seasonal baseline**, a **spatial propagation term** that mixes in delayed signals from upstream neighbors, and a **stochastic component** that includes both station-specific noise and discrete event perturbations.

### 4.1 Seasonal Baseline

Each station has its own intercept and amplitude, drawn at generation time and held fixed across the study period:

$$
b_n(t) = a_n + \beta_n^{\text{daily}} \sin\!\left(\tfrac{2\pi t}{24} + \phi_n^d\right) + \beta_n^{\text{weekly}} \sin\!\left(\tfrac{2\pi t}{168} + \phi_n^w\right)
$$

Here $a_n$ is the station's mean flow level, $\beta_n^{\text{daily}}$ and $\beta_n^{\text{weekly}}$ are amplitudes for the 24-hour and 168-hour cycles, and $\phi_n^d, \phi_n^w$ are random phase offsets that produce realistic station heterogeneity. All four are sampled once from prior distributions calibrated to PeMS-like ranges (mean flow ≈ 200 veh/h, daily amplitude ≈ 150 veh/h, weekly amplitude ≈ 50 veh/h).

### 4.2 Spatial Propagation

To inject controllable spatial dependence, we mix each node's baseline with a delayed weighted sum of its upstream neighbors:

$$
s_n(t) = (1 - \alpha)\, b_n(t) + \alpha \sum_{m \in \mathcal{U}(n)} w_{nm}\, b_m(t - \tau_{nm})
$$

where $\mathcal{U}(n)$ is the set of upstream neighbors of $n$, $w_{nm}$ are normalized edge weights ($\sum_m w_{nm} = 1$), $\tau_{nm}$ is an integer hour-lag drawn from $\{1, 2, 3\}$ to simulate physical propagation time between sensors, and $\alpha \in [0, 1]$ is the **spatial coupling parameter** ,the central knob of the study.

The interpretation of $\alpha$ is direct: at $\alpha = 0$, every station's signal is independent of every other station's, and there is no information for graph methods to exploit. At $\alpha = 1$, each station's signal is entirely determined by its upstream neighbors, and graph methods should have a maximal advantage.

### 4.3 Events and Noise

Real traffic exhibits discrete shocks; incidents, construction, weather, that propagate downstream. We inject events at random times by selecting a random origin node and a random duration $d \sim \text{Uniform}(2, 8)$ hours, then applying a multiplicative reduction $\eta \sim \text{Uniform}(0.3, 0.7)$ to the origin and its $K$-hop downstream descendants for the duration. Events do not propagate through $\alpha$; they are structurally separate so that the same event sequence can be reused across $\alpha$ values for a controlled comparison.

Finally, we add station-specific Gaussian noise:

$$
y_n(t) = \max\!\big(0,\ s_n(t) + e_n(t) + \varepsilon_n(t)\big), \qquad \varepsilon_n(t) \sim \mathcal{N}(0, \sigma_n^2)
$$

The non-negativity clip enforces the physical constraint that flow cannot be negative. The noise standard deviation $\sigma_n$ is set as a fraction (default 10%) of the station's mean flow.

### 4.4 Generator Parameters and Defaults

The generator has the following knobs, with default values used unless explicitly varied by an experiment:

| Symbol | Meaning | Default |
|---|---|---|
| $N$ | Number of nodes | 500 |
| $C$ | Number of corridors | 10 |
| $T$ | Number of hourly timestamps | 2,208 (≈ 92 days) |
| $K$ | Neighbors per side in adjacency | 2 |
| $\alpha$ | Spatial coupling strength | 0.5 |
| Daily amplitude | Mean of $\beta_n^{\text{daily}}$ | 150 |
| Weekly amplitude | Mean of $\beta_n^{\text{weekly}}$ | 50 |
| Noise level | $\sigma_n / a_n$ | 0.10 |
| Event rate | Events per node per study period | 0.5 |

The default $T$ matches the PeMS study period exactly (2024-10-01 to 2024-12-31, hourly). Choosing $N = 500$ rather than ≈1800 is a deliberate compromise between realism and iteration speed; a small-scale pilot at $N = 50$ is run first to validate that experiments produce sensible patterns before committing to the full-scale runs.

### 4.5 Assumptions and Limitations of the Generator

We make the following assumptions explicit so they can be examined in the discussion:

1. **Linear-additive structure.** Real traffic interactions are non-linear (congestion regimes, capacity constraints). The generator captures linear propagation only. This is conservative for graph methods — if anything, real non-linearities should *increase* the value of expressive non-linear models.
2. **Static graph topology.** The corridor structure does not change over time. PeMS is also effectively static at the timescales we consider.
3. **Stationary parameters.** Station amplitudes and means do not drift across the study period. Holiday weeks in PeMS do show drift; this is a limitation we acknowledge.
4. **No reverse propagation.** We model only upstream-to-downstream flow propagation. Backward propagation of congestion is real but harder to calibrate; including it would not change the conclusions of the spatial-coupling experiment.

These assumptions mean the generator is a **simplified surrogate** for real traffic, not a digital twin. Its purpose is to provide controlled conditions under which model behavior can be isolated, not to replicate PeMS dynamics.

## 5. Experimental Design

Each of the five experiments follows the same protocol. We generate data with a fixed random seed for the data-generating process, train each model with three independent training seeds (for reasons established by RQ3), and report mean ± std MAE across training seeds at horizons $h \in \{12, 24, 48, 72\}$ hours. Where an experiment varies a parameter (e.g. $\alpha$ in RQ1), we hold all other generator parameters at their defaults.

The protocol is summarized below. Implementation details for each experiment are documented in the corresponding section of this notebook.

| Experiment | Parameter Varied | Values | Models Evaluated |
|---|---|---|---|
| RQ1 | $\alpha$ (spatial coupling) | {0.0, 0.2, 0.5, 0.8} | All 8 |
| RQ2 | Outage type | {random, clustered, extended, intermittent, adversarial} | RF + 4 graph variants |
| RQ3 | Training seed | 5 seeds | 4 graph variants |
| RQ4 | Input length $L$ | {12, 24, 48, 168} | RF + 4 graph variants |
| RQ5 | Forecast horizon $h$ × architecture | {12, 24, 48, 72} × 4 variants | 4 graph variants |

All experiments share a common evaluation harness defined later in this notebook. Results are written to a structured `artifacts/sim/` directory, with each run producing a JSON metadata file, a metrics CSV, and a checkpoint, mirroring the artifact structure of the PeMS pipeline.

## 6. What This Study Cannot Establish

It is worth being explicit about the limits of what controlled simulation can tell us. The simulation study can establish that the proposed model is competitive **under conditions described by our generator's parameter ranges**. It cannot establish that those conditions are universal, nor that they exhaust the relevant axes of variation in real data. In particular, the simulation will not capture: heterogeneous sensor reliability characteristics, weather-induced demand shifts, holiday and seasonal regime changes, or driver behavioral responses to congestion. The validity of generalizing simulation conclusions to PeMS rests on the bridge experiment in Section X (to be added), which checks whether the spatial-coupling and other relevant parameters of the simulated regime where the proposed model wins are consistent with values estimated from PeMS data.

## 7. Notebook Organization

The remainder of this notebook is organized as follows. Section 8 implements the synthetic generator. Section 9 implements the evaluation harness and shared utilities. Sections 10–14 contain Experiments 1–5 corresponding to RQ1–RQ5. Section 15 aggregates results across experiments and discusses bridge inferences to PeMS. Section 16 produces the publication-ready figures and tables.